In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [2]:
df = pd.read_csv("../data/AirPassengers.csv")

df.head()

,Month,#Passengers
0,1949-01,112
1,1949-02,118
2,1949-03,132
3,1949-04,129
4,1949-05,121


In [3]:
data = df["#Passengers"].values

data[:10]

array([112, 118, 132, 129, 121, 135, 148, 148, 136, 119])

In [4]:
data = data.reshape(-1, 1)

data.shape

(144, 1)

In [5]:
scaler = MinMaxScaler()

data_scaled = scaler.fit_transform(data)

data_scaled[:5]

array([[0.01544402],
       [0.02702703],
       [0.05405405],
       [0.04826255],
       [0.03281853]])

In [6]:
joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

print("Scaler Saved")

Scaler Saved


In [7]:
window_size = 5

X = []
y = []

for i in range(
    window_size,
    len(data_scaled)
):
    
    X.append(
        data_scaled[
            i-window_size:i,
            0
        ]
    )
    
    y.append(
        data_scaled[
            i,
            0
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(139, 5)
(139,)


In [8]:
model = Sequential()

model.add(
    Dense(
        64,
        activation="relu",
        input_shape=(5,)
    )
)

model.add(
    Dense(
        32,
        activation="relu"
    )
)

model.add(
    Dense(
        1
    )
)

model.summary()

c:\Users\yanal\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,497 (9.75 KB)

 Trainable params: 2,497 (9.75 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer="adam",
    loss="mse"
)

In [10]:
history = model.fit(
    X,
    y,
    epochs=100,
    batch_size=8,
    verbose=1
)

Epoch 1/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0090   
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0067 
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0054 
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0049 
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0044 
Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0043 
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0046 
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0039 
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0039 
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0038 
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0040
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0038 
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0035 
Epoch 14/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0039 
Epoch 15/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - lo

In [11]:
model.save(
    "../models/model.keras"
)

print("Model Saved")

Model Saved


In [12]:
sample = np.array([
    [
        112,
        118,
        132,
        129,
        121
    ]
])

sample_scaled = scaler.transform(
    sample.reshape(-1,1)
)

sample_scaled = sample_scaled.reshape(
    1,
    5
)

prediction = model.predict(
    sample_scaled
)

result = scaler.inverse_transform(
    prediction.reshape(-1,1)
)

print(
    "Predicted Passengers:",
    result[0][0]
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
Predicted Passengers: 125.987625
